# 3주차 실습 — 신호 기준을 높이면 교체와 비용이 어떻게 달라지는가

교재와 같은 Backblaze 2026년 1분기 기록으로 교체 기준을 비교해 보자. 대기 불량 섹터가 0보다 크면 교체하는 규칙에서 출발해, 기준값을 높일 때 교체 부담과 예방 대상 고장이 어떻게 달라지는지 확인한다.

> **제시한 교체 기준 중 비용 조건에 따라 무엇을 선택하겠는가?**

준비 코드를 실행하고 과제 1부터 차례로 진행한다. 각 과제에서 만든 결과를 다음 과제에 쓴다. 약 60분으로, 준비 5분, 과제 1~3 각 5분, 과제 4는 10분, 과제 5는 5분, 과제 6은 10분, 과제 7은 15분을 쓴다. 약 523만 행을 처음 읽을 때는 다운로드 시간이 추가될 수 있다.

기준값은 0·1·4·16을 비교한다. 신호가 조금만 있어도 교체하는 경우와 더 큰 값에서 교체하는 경우의 차이를 보기 위한 후보이며, 이 값이 현장 권고 기준이라는 뜻은 아니다. 비용 비율도 5·20·50의 가정이다. 제공된 후보와 비용 조건 안에서 선택하고, 다른 기준이나 비용에서도 같은 선택이라고 확대하지 않는다.

매일 신호를 확인해 조건에 처음 걸린 날 교체한다고 본다. 처음 관측부터 조건에 걸린 디스크도 전체 교체에 포함하며 한 디스크는 한 번만 센다. **가정:** 확인한 날 교체를 완료하고 교체 디스크의 관측 고장을 막으며, 새 디스크는 남은 기간에 고장 나거나 다시 교체되지 않는다. 고장 당일에 처음 신호가 켜진 디스크는 예방 대상으로 포함하지 않는다. ‘예방 대상 고장’은 이 가정 아래의 계산이며, 실제 교체 실험으로 입증한 효과는 아니다.

**과제 1~6의 확인 셀은 작성 여부와 자료형·표의 모양만 확인한다.** “형식 확인 완료”는 정답이라는 뜻이 아니다. 직접 결과를 출력하고 계산을 점검한다. 계산값은 제출 후 교수자가 채점하므로 반올림하지 않고 저장한다. 과제 7은 수치와 본인의 판단을 문장으로 작성한다.

## 준비 — 고장 전에 판단할 수 있었던 기록

원자료는 2026년 1월 1일~3월 31일의 90일 동안 기록한 디스크 6만 개 표본이다. 한 행은 디스크 한 개의 하루 상태다. 교재처럼 누적 가동 시간이 0 이상인 기록을 사용한다.

아래 코드는 자료 읽기와 고장 전 기록 준비를 제공한다. 고장 날짜를 기록에 대응시켜 그날보다 앞선 행을 남긴다. 고장이 기록되지 않은 디스크는 관측된 기록을 모두 남긴다. 고장 날짜는 과거 규칙을 평가할 때 사용하고, 교체 조건은 당시 관측된 신호로 판단한다.

| 준비된 이름 | 뜻 |
|---|---|
| `before` | 고장 전 기록과 고장 미기록 디스크의 기록. `serial_number`는 디스크 번호, `pending_sectors`는 대기 불량 섹터 수 |
| `first` | 디스크별 첫 관측. 처음부터 교체 조건에 걸린 경우를 확인할 때 사용 |
| `failed` | 이 기간에 고장이 기록된 디스크 번호 집합 |

교재 7-4·8-1·8-2의 계산이다. 준비 셀을 실행해 고장 전 기록 수와 디스크 수, 관측 고장 수를 확인한다.

In [42]:
# [셋업] 환경과 열 이름 — 수정하지 않는다.
import os
import sys
import numpy as np
import pandas as pd
for _p in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p)
        break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py
from balab import load, checker, summary, check_format
check = checker("week03")
RENAME = {"smart_9_raw": "power_on_hours", "smart_5_raw": "reallocated_sectors",
          "smart_187_raw": "uncorrectable_errors", "smart_188_raw": "command_timeouts",
          "smart_197_raw": "pending_sectors", "smart_198_raw": "offline_uncorrectable"}
bb = load("backblaze_2026").rename(columns=RENAME)
bb = bb[bb["power_on_hours"] >= 0].copy()
failed = set(bb.loc[bb["failure"] == 1, "serial_number"])
fail_date = bb[bb["failure"] == 1].set_index("serial_number")["date"]
fdate = bb["serial_number"].map(fail_date)
before = bb[fdate.isna() | (bb["date"] < fdate)]
first = before.sort_values("date").groupby("serial_number").head(1)
len(before), before["serial_number"].nunique(), len(failed)

(5221600, 59927, 194)

## 과제 1 — 교체할 디스크를 모은다

**대기 불량 섹터가 있으면 교체할 디스크를 찾아보자.**

`before`에서 대기 불량 섹터가 0보다 큰 기록의 디스크 번호를 집합 `hit0`에 담는다. 같은 디스크의 신호가 여러 날 켜져 있어도 집합에는 한 번만 들어간다. 처음 관측부터 신호가 있던 디스크도 제외하지 않는다.

교재 8-4의 조건 필터·`loc`·`set`을 사용한다. 결과는 디스크 번호 집합이다. `len`으로 전체 교체 개수를 확인한다. 확인 셀은 디스크 번호를 문자열 집합으로 작성했는지 확인한다.

In [43]:
hit0 = set(before.loc[before["pending_sectors"] > 0, "serial_number"])  # TODO: 고장 전 pending_sectors가 0보다 큰 디스크 번호 집합
# 데이터포인트가 아니라 디스크 번호를 모은다 (loc, set, len)

In [44]:
# [확인] 과제 1 — 수정하지 않는다.
if os.environ.get("BALAB_ANSWER_KEY"):
    if hit0 is None:
        check(1, n0=None)
    else:
        check(1, n0=len(hit0))
else:
    check_format(1, hit0=hit0)

형식 확인 완료: 과제 1 — 계산값의 정답 여부는 제출 후 채점합니다.


## 과제 2 — 교체 대상과 관측 고장이 겹치는지 확인한다

**교체할 디스크 중 실제로 고장이 기록된 디스크를 찾아보자.**

`hit0`와 `failed`의 교집합을 `prevented0`에 담는다. 즉시 교체로 그 고장을 막을 수 있다는 가정 아래 예방 대상으로 포함한 디스크다.

교재 8-3처럼 `&`로 교집합을 구하고 `len`으로 개수를 확인한다. 입력과 결과 모두 디스크 번호 집합이다. 확인 셀은 결과가 문자열 집합인지 확인한다.

In [45]:
prevented0 = hit0 & failed  # TODO: 교체 집합 hit0와 관측 고장 집합 failed의 교집합
# 두 집합의 공통 디스크를 센다 (교집합, len)

In [46]:
# [확인] 과제 2 — 수정하지 않는다.
if os.environ.get("BALAB_ANSWER_KEY"):
    if prevented0 is None:
        check(2, m0=None)
    else:
        check(2, m0=len(prevented0))
else:
    check_format(2, prevented0=prevented0)

형식 확인 완료: 과제 2 — 계산값의 정답 여부는 제출 후 채점합니다.


## 과제 3 — 고장 하나당 교체 부담을 계산한다

**고장 하나를 예방 대상으로 포함하려면 몇 개를 교체하는지 계산해 보자.**

`hit0`의 크기를 `n0`, `prevented0`의 크기를 `m0`에 담고, 전체 교체를 예방 대상 고장으로 나눈 값을 `pf0`에 담는다. 교재 8-4·8-7의 계산이다.

결과는 정수 `n0`·`m0`와 실수 `pf0`다. 세 값을 나란히 출력한다. 이 규칙에는 예방 대상 고장이 있어 나눗셈이 가능하다. 확인 셀은 개수가 정수이고 비율이 숫자인지 확인한다.

In [47]:
n0 = len(hit0)  # TODO: 전체 교체 개수
m0 = len(prevented0)  # TODO: 예방 대상 고장 개수
pf0 = n0 / m0  # TODO: 고장 하나당 전체 교체 개수
# 두 집합의 크기로 계산한다 (len, 나누기)

In [48]:
# [확인] 과제 3 — 수정하지 않는다.
if os.environ.get("BALAB_ANSWER_KEY"):
    if n0 is None or m0 is None or pf0 is None:
        check(3, n0=None)
    else:
        check(3, n0=n0, m0=m0, pf0=(pf0, 0.0001))
else:
    check_format(3, n0=n0, m0=m0, pf0=pf0)

형식 확인 완료: 과제 3 — 계산값의 정답 여부는 제출 후 채점합니다.


## 과제 4 — 다른 기준값에서도 교체와 고장을 비교한다

**신호 기준을 높이면 교체 대상과 예방 대상 고장이 얼마나 줄어드는지 비교해 보자.**

기준값 0·1·4·16에 대해 과제 1~2를 반복한다. 대기 불량 섹터가 기준값보다 큰 기록을 사용하며 같은 값은 포함하지 않는다.

먼저 반복문에서 규칙 이름 `rule`, 전체 교체 `replaced`, 예방 대상 고장 `prevented`를 사전으로 만들어 목록 `rows`에 모은다. 교재 8-6의 `for`·`set`·교집합·`append`를 사용한다. 목록을 모은 뒤 다음 셀에서 표를 만든다.

In [49]:
rows = []  # TODO: 기준값별 rule·replaced·prevented를 모은 목록
# 각 기준값에서 집합을 만들고 전체 교체와 교집합 크기를 담는다 (for, set, len, append)
for thr in [0, 1, 4, 16]:
    hit = set(before.loc[before["pending_sectors"] > thr, "serial_number"])
    prevented = hit & failed
    rows.append({"rule": f"pending > {thr}", "replaced": len(hit), "prevented": len(prevented)})

**모은 결과를 표로 놓고 두 개수를 함께 읽어 보자.**

`rows`를 `DataFrame`으로 바꾸고 `set_index`로 `rule`을 인덱스로 두어 `threshold_tbl`에 담는다. 결과는 4행 2열 표다. 규칙 이름은 pending > 0, pending > 1, pending > 4, pending > 16이다. 기준값 0의 결과는 과제 3과 같아야 한다.

확인 셀은 4행 2열의 이름과 숫자 자료형을 확인한다.

In [50]:
threshold_tbl =  pd.DataFrame(rows).set_index("rule")  # TODO: rows를 표로 바꾸고 rule을 인덱스로 둔 비교표
# replaced·prevented 두 열을 비교한다 (DataFrame, set_index)

In [51]:
# [확인] 과제 4 — 수정하지 않는다.
if os.environ.get("BALAB_ANSWER_KEY"):
    if threshold_tbl is None:
        check(4, n_rules=None)
    else:
        check(4, n_rules=len(threshold_tbl), replaced=threshold_tbl["replaced"], prevented=threshold_tbl["prevented"])
else:
    check_format(4, threshold_tbl=threshold_tbl)

형식 확인 완료: 과제 4 — 계산값의 정답 여부는 제출 후 채점합니다.


## 과제 5 — 비용 조건 하나에서 비교한다

**고장 후 대응이 계획 교체보다 20배 비싸다면 어느 규칙이 유리한지 계산해 보자.**

계획 교체 한 번의 비용을 1로 둔다. 규칙의 총비용은 전체 교체 비용과 남은 고장의 대응 비용을 더한 것이다. 남은 고장은 관측 고장 전체에서 예방 대상 고장을 뺀 개수다. 고장 후 교체만 하면 관측 고장 전체에 20배의 비용이 든다.

교재 9-3처럼 규칙의 총비용을 고장 후 교체 비용으로 나눠 상대 비용을 계산한다. `DataFrame`에 `R=20` 열로 담아 `costs`를 만들고, `loc`으로 after failure 행을 추가해 1을 넣는다. 결과는 5행 1열 표다. 확인 셀은 다섯 규칙의 행 이름과 R=20 숫자 열을 확인한다.

In [52]:
costs = pd.DataFrame({"R=20": (threshold_tbl["replaced"] + (len(failed) - threshold_tbl["prevented"]) * 20) / (len(failed) * 20)})  # TODO: 규칙별 R=20 상대 비용과 after failure 행을 담은 표
# 전체 교체 비용과 남은 고장 비용을 더해 고장 후 교체 비용으로 나눈다 (DataFrame, loc)
costs.loc["after failure"] = 1

In [53]:
# [확인] 과제 5 — 수정하지 않는다.
if os.environ.get("BALAB_ANSWER_KEY"):
    if costs is None:
        check(5, n_choices=None)
    else:
        check(5, n_choices=len(costs), cost20=(costs["R=20"], 0.0001))
else:
    check_format(5, costs=costs)

형식 확인 완료: 과제 5 — 계산값의 정답 여부는 제출 후 채점합니다.


## 과제 6 — 비용 조건을 바꿔 선택을 비교한다

**고장 후 대응 비용이 달라져도 같은 규칙을 선택할지 비교해 보자.**

비용 비율 5와 50에서도 과제 5의 상대 비용을 구해 `costs`에 `R=5`·`R=50` 열을 추가한다. after failure 행은 각 열에서 1이다. 교재 9-3처럼 `for`로 두 비율의 계산을 반복할 수 있다.

세 비용 비율의 상대 비용이 담긴 표를 출력한다. 후보를 바꾸거나 처음부터 신호가 있었던 디스크의 교체 비용을 빼지 않는다.

In [54]:
# TODO: costs에 R=5·R=50의 상대 비용 열을 추가한다
# 두 비율에서 과제 5를 반복하고 after failure 행은 1로 둔다 (for, loc)
for R in [5, 50]:
    costs[f"R={R}"] = (threshold_tbl["replaced"] + (len(failed) - threshold_tbl["prevented"]) * R) / (len(failed) * R)
    costs.loc["after failure", f"R={R}"] = 1

**각 비용 조건에서 가장 싼 규칙의 이름을 읽어 보자.**

`sort_values`로 해당 비용 열을 작은 순서로 정렬한다. 정렬한 표의 `index`에서 첫 값이 가장 싼 규칙 이름이다. 비용 비율 5·20·50의 선택을 `best5`·`best20`·`best50`에 담는다. 이 세 조건에서는 최저 비용에 동률이 없다.

결과는 문자열 세 개다. 확인 셀은 다섯 규칙의 행 이름·세 비용 열의 숫자 자료형과 선택 결과가 문자열인지 확인한다.

In [55]:
best5 = costs["R=5"].sort_values().index[0]  # TODO: 비용 비율 5에서 가장 싼 규칙 이름
best20 = costs["R=20"].sort_values().index[0]  # TODO: 비용 비율 20에서 가장 싼 규칙 이름
best50 = costs["R=50"].sort_values().index[0]  # TODO: 비용 비율 50에서 가장 싼 규칙 이름
# 각 열로 정렬한 표의 첫 인덱스 값을 읽는다 (sort_values, index)

In [56]:
# [확인] 과제 6 — 수정하지 않는다.
if os.environ.get("BALAB_ANSWER_KEY"):
    if costs is None or best5 is None or best20 is None or best50 is None or "R=5" not in costs.columns or "R=50" not in costs.columns:
        check(6, best5=None)
    else:
        check(6, cost5=(costs["R=5"], 0.0001), cost20=(costs["R=20"], 0.0001), cost50=(costs["R=50"], 0.0001), best5=best5, best20=best20, best50=best50)
else:
    check_format(6, costs=costs, best5=best5, best20=best20, best50=best50)

형식 확인 완료: 과제 6 — 계산값의 정답 여부는 제출 후 채점합니다.


## 과제 7 — 본인의 판단과 설명을 쓴다

**운영 담당자라면 어떤 조건에서 어느 규칙을 선택할지 설명해 보자.**

아래 내용을 연결해 본인의 판단을 작성한다. 특정 규칙을 정답으로 요구하지 않으며 계산한 수치와 판단이 맞아야 한다.

1. 기준값이 다른 두 규칙을 골라 교체 개수와 예방 대상 고장을 비교한다. 교체 부담을 줄이는 대신 무엇을 놓치는지 설명한다.
2. 비용 비율이 다른 두 경우에서 선택한 규칙과 상대 비용을 적고, 선택이 달라지는 이유를 설명한다. 고장 하나당 교체가 적다는 사실만으로 가장 싼 규칙을 고를 수 있는지도 생각한다.
3. 본인의 적용 또는 보류 판단을 쓰고, 실제 운영 전에 확인할 정보 하나를 제시한다. 교체 소요 시간이나 새 디스크의 고장, 90일 이후 결과 중 필요한 것을 골라도 된다.

이는 관측한 90일에 규칙을 도입했다는 가정의 비교다. 처음부터 신호가 켜진 디스크도 교체하므로 비용에 포함하며, 비용을 빼면서 그 디스크의 관측 고장을 예방 효과에 남기지 않는다. 아직 관측되지 않은 고장이나 다른 비용 비율에서도 같은 선택이라고 단정하지 않는다.

**자동 채점하지 않는 서술 과제다.** 아래 셀을 편집해 수치와 설명을 함께 작성한다.

### 나의 판단과 설명

(두 규칙의 비교, 비용 조건별 선택, 본인의 판단과 추가 확인 사항을 문장으로 작성한다.)



## 제출 전 확인

아래 확인은 계산 과제 1~6의 작성 여부와 형식에 대한 것이다. 계산값의 정답 여부는 제출 후 채점한다. 형식 확인이 끝나도 과제 7의 설명을 작성해야 한다. 위에서부터 실행하고 본인의 계산과 설명이 담긴 노트북을 저장해 제출한다.

In [41]:
# 계산 과제 1~6 확인. 과제 7의 설명은 별도로 작성한다.
summary()

계산 과제 1~6 형식 확인 완료. 정답 판정은 아닙니다. 서술 답안도 작성해 제출하세요.


True